# I tuned my parameters. Is the result real?

Try enough settings of a strategy and the best one looks profitable, even on a
market where no strategy can win. [`strategy_test`](../functions_fin/strategy_test.md)
catches this: it permutes the market's bars and reruns your whole procedure,
search included, on every copy. If the search does as well on permuted markets,
the result is luck.

The data is a random walk, so we know there is no edge to find.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import (RollingZscore, BandSignal, BacktestOHLCTarget, position_test,
                      strategy_test, deflated_sharpe)

T, SUB, PPY = 2000, 6, 24 * 365   # 2000 hourly bars of 6 steps; bars per year


def random_walk(seed):
    # hourly OHLC bars of a random walk with 1% volatility per bar
    rng = np.random.default_rng(seed)
    p = 100 * np.exp(np.cumsum(rng.standard_normal(T * SUB) * 0.01 / np.sqrt(SUB)))
    p = p.reshape(T, SUB)
    c = p[:, -1]
    o = np.r_[p[0, 0], c[:-1]]
    return o, np.maximum(p.max(axis=1), o), np.minimum(p.min(axis=1), o), c


o, h, l, c = random_walk(7)

## The tuned strategy

A z-score band rule ([`BandSignal`](../functions_signal/BandSignal.md) on a
[`RollingZscore`](../functions_rolling/RollingZscore.md)) with 30 settings:
6 windows and 5 entry levels. `tuned` runs all of them and keeps the one with
the best Sharpe ratio, as a parameter search does.

In [ ]:
engine = BacktestOHLCTarget(taker_fee=5e-4)


def settings(c):
    # one target column per setting, (T, 30)
    cols = [BandSignal(entry=k, exit=0.0)(RollingZscore(w)(np.log(c)))
            for w in (10, 20, 30, 50, 80, 120) for k in (0.5, 1.0, 1.5, 2.0, 2.5)]
    return np.nan_to_num(np.column_stack(cols))


def sharpe(out):
    # annualized Sharpe ratio of each backtest; NaN for a setting that never trades
    pnl = out[..., 1]
    sd = pnl.std(axis=0, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(sd > 0, pnl.mean(axis=0) / sd, np.nan) * np.sqrt(PPY)


def tuned(o, h, l, c):
    runs = engine(settings(c), o, h, l, c)
    return runs[:, int(np.nanargmax(sharpe(runs)))]


print(f"best of 30 settings: Sharpe {sharpe(tuned(o, h, l, c)):.2f}")

## Test the whole search

In [ ]:
whole = strategy_test(tuned, o, h, l, c, n=499, periods_per_year=PPY)

best = int(np.nanargmax(sharpe(engine(settings(c), o, h, l, c))))
chosen = position_test(engine, settings(c)[:, best], o, h, l, c, n=999, periods_per_year=PPY)

print(f"strategy_test, whole search:    p = {whole.p_value:.3f}")
print(f"position_test, chosen setting:  p = {chosen.p_value:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, res, title in [(axes[0], whole, "strategy_test: search on permuted markets"),
                       (axes[1], chosen, "position_test: chosen setting, random timing")]:
    ax.scatter(res.null_stats["sharpe"], res.null_stats["max_drawdown"], s=6,
               color="lightsteelblue")
    ax.scatter(res.observed_stats["sharpe"], res.observed_stats["max_drawdown"], s=250,
               marker="*", color="crimson", label=f"real run, p = {res.p_value:.3f}")
    ax.set_title(title, fontsize=10); ax.set_xlabel("Sharpe ratio"); ax.legend(fontsize=8)
axes[0].set_ylabel("max drawdown ($)")
plt.tight_layout()

On the left, the search finds Sharpe ratios like the real one on many permuted
markets: the star sits inside the cloud. On the right, `position_test` compares
the chosen setting only with random timings of itself. It cannot see that this
setting was picked because it fit this path, so it gives a lower p-value than
it should.

## Cheaper checks of a search

Two checks need no rerun of the strategy. `position_test` with all settings as
a `(T, 30)` target returns `family_p_value`, which compares the chosen setting
with the best of the 30 in every round. `deflated_sharpe` raises the bar to the
Sharpe ratio the best of the 30 would reach by luck.

In [ ]:
runs = engine(settings(c), o, h, l, c)
family = position_test(engine, settings(c), o, h, l, c, n=999, periods_per_year=PPY)
trials = sharpe(runs) / np.sqrt(PPY)                # per-bar Sharpe of every setting
prob, _ = deflated_sharpe(runs[:, best], trials[np.isfinite(trials)])
print(f"family p = {family.family_p_value[best]:.3f}, deflated Sharpe P(edge) = {prob:.3f}")

Use these when the search is a plain choice among fixed variants. Use
`strategy_test` when the procedure does more than choose, for example when it
fits a model, filters, or adapts its parameters over time.

## How often is each test fooled?

The same experiment on 60 independent random walks. Where nothing is there, a
sound test gives p below 0.05 in about 5% of the walks.

In [ ]:
p_whole, p_chosen = [], []
for k in range(60):
    o, h, l, c = random_walk(100 + k)
    best = int(np.nanargmax(sharpe(engine(settings(c), o, h, l, c))))
    p_whole.append(strategy_test(tuned, o, h, l, c, n=199, periods_per_year=PPY,
                                 seed=k).p_value)
    p_chosen.append(position_test(engine, settings(c)[:, best], o, h, l, c, n=199,
                                  periods_per_year=PPY, seed=k).p_value)

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, p, title in [(axes[0], p_whole, "strategy_test, whole search"),
                     (axes[1], p_chosen, "position_test, chosen setting")]:
    ax.hist(p, bins=np.linspace(0, 1, 21), color="lightsteelblue")
    ax.axvline(0.05, color="crimson", lw=1)
    ax.set_title(f"{title}: p < 0.05 in {np.mean(np.array(p) < 0.05):.0%} of walks",
                 fontsize=10)
    ax.set_xlabel("p-value")
axes[0].set_ylabel("random walks")
plt.tight_layout()

## Adapt it

- `strategy_test(strategy, *inputs)` calls `strategy(*inputs)` on the real data
  and on each permuted copy. The strategy returns a backtest engine's `(T, 4)`
  output; a compiled `Pipeline` works too.
- `method="bars"` (the default) permutes the OHLC inputs at positions `ohlc`;
  `method="returns"` permutes a single price input at position `price`.
- Use `mode="diff"` for prices that can be zero or negative, and list inputs
  that must not move, such as timestamps, in `fixed`.
- Each rerun runs your whole strategy, so start with a small `n` and raise it
  for the final number.